# Finding optimal Q in a charger grid

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkipngenokoech/Tabular-Q-learning/blob/main/charger_grid.ipynb)

The robot starts at A. Reaching the charger at D earns +10 and ends the episode. Every other move earns 0. A move into a wall leaves the robot in the same cell.

| A (start) | B |
| --- | --- |
| C | D (charger) |

In Google Colab, choose Runtime → Run all. The notebook shows the starting table, each move in the two guided episodes, and one table per episode in the training run.

In [ ]:
from importlib.util import find_spec

if find_spec('gymnasium') is None:
    %pip install -q gymnasium

## 1. Set up the Gymnasium environment

In [ ]:
import random

import gymnasium as gym

CELLS = ("A", "B", "C", "D")
ACTIONS = ("Up", "Down", "Left", "Right")
# Each row lists the destination for Up, Down, Left, Right.
DESTINATIONS = ((0, 2, 0, 1), (1, 3, 0, 1), (0, 2, 2, 3))
ALPHA = 0.5
GAMMA = 0.9


class ChargerGrid(gym.Env):
    def __init__(self):
        self.observation_space = gym.spaces.Discrete(4)
        self.action_space = gym.spaces.Discrete(4)
        self.state = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0
        return self.state, {}

    def step(self, action):
        self.state = DESTINATIONS[self.state][action]
        terminated = self.state == 3
        reward = 10 if terminated else 0
        return self.state, reward, terminated, False, {}

## 2. Update one entry in the Q-table

The target is the reward just earned plus the best estimate in the next state's row, discounted by 0.9. At the charger, the future term is zero. The learning rate of 0.5 moves the old estimate halfway toward the target.

In [ ]:
def update(q, state, action, reward, next_state, terminated):
    best_next = 0.0 if terminated else max(q[next_state])
    target = reward + GAMMA * best_next
    old = q[state][action]
    q[state][action] = old + ALPHA * (target - old)


from html import escape
from IPython.display import HTML, display


def show_table(q, heading):
    columns = "".join(f'<th scope="col">{action}</th>' for action in ACTIONS)
    rows = []
    for cell, values in zip(CELLS, [*q, ["—"] * len(ACTIONS)]):
        entries = "".join(
            f"<td>{value:.2f}</td>" if isinstance(value, (float, int))
            else f"<td>{value}</td>"
            for value in values
        )
        rows.append(f'<tr><th scope="row">{cell}</th>{entries}</tr>')

    display(HTML(
        f"<table><caption><strong>{escape(heading)}</strong></caption>"
        f'<thead><tr><th scope="col">Cell</th>{columns}</tr></thead>'
        f"<tbody>{''.join(rows)}</tbody></table>"
    ))

## 3. Trace A → B → D twice

Before each move, predict which cell will change. In episode 1, the first visit to B has no learned value yet. In episode 2, A/Right can use the value learned for B/Down.

In [ ]:
env = ChargerGrid()
q = [[0.0] * len(ACTIONS) for _ in CELLS[:3]]
show_table(q, "Before training")

for episode in (1, 2):
    state, _ = env.reset()
    for action in (ACTIONS.index("Right"), ACTIONS.index("Down")):
        next_state, reward, terminated, _, _ = env.step(action)
        update(q, state, action, reward, next_state, terminated)
        show_table(q, f"Episode {episode} | {CELLS[state]} → {CELLS[next_state]} | Reward {reward}")
        state = next_state

assert q[0][3] == 2.25 and q[1][1] == 7.5
env.close()

## 4. Let the robot choose moves

The first two episodes replay the same route. After that, the robot chooses randomly with probability epsilon and otherwise picks a highest-valued action, breaking ties randomly. Each run starts with a fresh Q-table. Try `EPSILON = 0.0` and then `EPSILON = 0.5` with `EPISODES = 20` and compare row C.

In [ ]:
def choose_action(values, epsilon, rng):
    if rng.random() < epsilon:
        return rng.randrange(len(values))
    best = max(values)
    return rng.choice([action for action, value in enumerate(values)
                       if value == best])


def main(episodes, epsilon):
    env = gym.wrappers.TimeLimit(ChargerGrid(), max_episode_steps=50)
    rng = random.Random(7)
    q = [[0.0] * len(ACTIONS) for _ in CELLS[:3]]
    show_table(q, "Before training")

    for episode in range(1, episodes + 1):
        state, _ = env.reset()
        route = [CELLS[state]]
        total_reward = 0

        while True:
            # Replay Right, Down twice so we can check the hand calculations.
            if episode <= 2:
                action = (3, 1)[len(route) - 1]
            else:
                action = choose_action(q[state], epsilon, rng)

            next_state, reward, terminated, truncated, _ = env.step(action)
            update(q, state, action, reward, next_state, terminated)
            route.append(CELLS[next_state])
            total_reward += reward
            state = next_state
            if terminated or truncated:
                break

        if episode == 1:
            assert q[0][3] == 0 and q[1][1] == 5
        if episode == 2:
            assert q[0][3] == 2.25 and q[1][1] == 7.5
        show_table(q, f"Episode {episode} | {' -> '.join(route)} | Reward {total_reward}")

    env.close()

In [ ]:
EPISODES = 6
EPSILON = 0.2
main(EPISODES, EPSILON)